# CodSoft Week 2 — Sales Data Analysis

An end-to-end sales analysis using the supplied **fictional practice workbook**. The notebook calls the same tested functions as `run_analysis.py`, keeping analysis, exports, and the Power BI handoff consistent.

**The source does not specify a currency.** Amounts are displayed as source units; no currency is assumed. The original workbook is never modified.

## 1. Set up the project

Install dependencies with `python -m pip install -r requirements.txt`, open this notebook from the project, select the project's Python kernel, and run all cells from top to bottom.

In [ ]:
from pathlib import Path
import sys

candidate_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next(
    (
        root
        for root in candidate_roots
        if (root / "src" / "sales_analysis.py").is_file()
        and (root / "data" / "raw" / "Sales_Data_Raw_Practice.xlsx").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open the notebook from the project folder or one of its subfolders."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Sales_Data_Raw_Practice.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
print(f"Project folder: {PROJECT_ROOT}")
print(f"Original dataset: {DATA_PATH}")

## 2. Inspect the source data

Review the original headers, example records, blanks, value types, and exact duplicates before cleaning. These checks do not change the source workbook.

In [ ]:
import pandas as pd
from IPython.display import display

raw_sales = pd.read_excel(
    DATA_PATH,
    sheet_name="Raw_Sales_Data",
    engine="openpyxl",
)
print(f"Source records: {len(raw_sales):,}")
print(f"Source columns: {', '.join(raw_sales.columns.astype(str))}")
print(f"Exact duplicate rows: {int(raw_sales.duplicated().sum()):,}")
display(raw_sales.head(10))

source_quality = pd.DataFrame(
    {
        "Missing source values": raw_sales.isna().sum(),
        "Source value types": raw_sales.apply(
            lambda column: ", ".join(
                sorted({type(value).__name__ for value in column.dropna()})
            )
        ),
        "Distinct nonblank values": raw_sales.nunique(dropna=True),
    }
).sort_values("Missing source values", ascending=False)
display(source_quality)

## 3. Clean and analyze the dataset

The shared cleaning functions standardize text, parse dates and numeric values, and document decisions. Invalid values are reported rather than guessed, blank business measures remain unknown instead of becoming zero, and suspicious return-like values are retained for review. This cell generates the cleaned CSVs, Excel workbook, business findings, and charts in `outputs/`.

In [ ]:
from src.sales_analysis import (
    build_analysis_tables,
    build_insights,
    build_kpi_summary,
    generate_outputs,
    load_and_clean_data,
)

cleaned_sales, quality_report = load_and_clean_data(DATA_PATH)
analysis_tables = build_analysis_tables(cleaned_sales)
kpi_summary = build_kpi_summary(cleaned_sales, quality_report)
output_files = generate_outputs(cleaned_sales, quality_report, OUTPUT_DIR)

print(f"Source records: {len(raw_sales):,}")
print(f"Cleaned records: {len(cleaned_sales):,}")
print(f"Output folder: {OUTPUT_DIR}")
display(quality_report)

## 4. Key performance indicators

Order counts use distinct Order IDs, the customer total excludes unknown customer names, and profit margin is total reported profit divided by total reported sales. Currency is not specified.

In [ ]:
display(kpi_summary)

## 5. Sales trends over time

Transactions with missing or invalid order dates are retained in overall known sales but cannot be included in monthly, quarterly, or annual comparisons. Growth compares adjacent periods and remains blank when a prior-period comparison cannot be calculated.

In [ ]:
monthly_trends = analysis_tables["monthly"]
quarterly_trends = analysis_tables["quarterly"]
yearly_trends = analysis_tables["yearly"]

print("Latest monthly periods")
display(monthly_trends.tail(12))
print("Quarterly sales and growth")
display(quarterly_trends)
print("Annual sales and growth")
display(yearly_trends)

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(OUTPUT_DIR / "charts" / "monthly_sales.png")))

## 6. Product and category performance

Compare product and category sales, profit, orders, units, and margin. A high-sales product is not necessarily the most profitable.

In [ ]:
display(analysis_tables["categories"])
display(analysis_tables["products"].head(10))
display(Image(filename=str(OUTPUT_DIR / "charts" / "sales_by_category.png")))
display(Image(filename=str(OUTPUT_DIR / "charts" / "top_10_products.png")))

## 7. Regional and customer performance

Customer order counts are distinct orders. Transactions with `Unknown` customers still contribute to sales totals but are not counted as identified customers.

In [ ]:
display(analysis_tables["regions"])
display(analysis_tables["customers"].head(10))
display(Image(filename=str(OUTPUT_DIR / "charts" / "sales_by_region.png")))
display(Image(filename=str(OUTPUT_DIR / "charts" / "top_10_customers.png")))

## 8. Data-derived findings and recommendations

The following report is generated from the supplied data, not example figures. Review the data-quality warnings and confirm business context before presenting any recommendation.

In [ ]:
report_text = build_insights(
    cleaned_sales,
    quality_report,
    analysis_tables,
    kpi_summary,
)
print(report_text)

## 9. Power BI and internship submission

- Load `outputs/cleaned_sales.csv` into Power BI Desktop and follow [`powerbi/README.md`](../powerbi/README.md).
- Create the provided DAX Calendar table and relationship before using time-intelligence measures.
- Add KPI cards, product/category and regional analyses, customer analysis, date drill-down, and report slicers; test their interactions.
- Create and save the interactive Power BI report as `powerbi/Sales_Dashboard.pbix` from Power BI Desktop. Capture screenshots of the completed report pages.
- Confirm all displayed results against `outputs/kpi_summary.csv` and the single `Dashboard` worksheet in `outputs/Sales_Analysis.xlsx`.
- Do not assume a currency or the meaning of negative sales and quantities; confirm them before making business recommendations.